# AI 2 · 3주차 실제 제품 사진 실험

새우깡 외에 **콜라 캔, 컵라면, 닭가슴살**을 실제로 찍은 사진으로 전처리와 자동 처리(`run_pipeline`)를 시험합니다.

알아보려는 것은 두 가지입니다.
1. 새우깡으로 정한 **재촬영 판단 기준**(흔들림, 너무 멂)이 다른 제품에서도 맞는지. 맞지 않으면 OCR 확신도를 보고 기준을 다시 정합니다.
2. 2주차에 계산으로만 구한 **자동 처리 개선판**의 오류율을 직접 측정하기

| 방식 | 설명 |
|---|---|
| `none` | 전처리 없음 |
| `auto` | 실제 서비스 흐름(`pipeline.run_pipeline`): 품질 진단으로 전처리를 고르고, OCR 확신도가 낮을 때만 회전·뒤집힘을 다시 확인 |

**실행 방법:** 메뉴에서 **런타임 → 모두 실행**을 누르면 됩니다. 사진 43장 기준 **1시간에서 1시간 30분** 걸립니다.

- 노트북은 충전기를 꽂고, 화면이 꺼지거나 절전되지 않게 해 주세요.
- 끝나면 마지막 셀에서 `ai2_week3_paddle.zip`이 내려받아집니다.

## 1. 레포 받아오기

In [ ]:
REPO_URL = "https://github.com/DL-2026F-Team05/package-ocr-preprocessing.git"
OCR_ENGINE = "paddle"

import os, subprocess, sys

os.chdir("/content")
if not os.path.exists("package-ocr"):
    if subprocess.run(["git", "clone", "-q", REPO_URL, "package-ocr"]).returncode != 0:
        raise RuntimeError("레포를 받아오지 못했습니다. REPO_URL을 확인하세요.")
os.chdir("/content/package-ocr")
subprocess.run(["git", "pull", "-q"])
print("현재 위치:", os.getcwd())

## 2. 패키지 설치

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "paddlepaddle", "paddleocr", "koreanize-matplotlib"], check=True)
print("설치 완료")

## 3. 사진과 정답 라벨 확인

새우깡(p01)만 2주차처럼 코드로 저조도·흔들림·회전·노이즈 사진을 만들고, 나머지 제품은 실제로 찍은 사진만 씁니다. 제품 번호는 `data/README.md`에 정리되어 있습니다.

In [ ]:
from collections import Counter
from pathlib import Path
import pandas as pd

sys.path.insert(0, "preprocessing")
import experiment, preprocess, pipeline

RAW, GEN, OUT = Path("data/raw"), Path("data/generated"), Path("results/ai2_week3/paddle")
labels = experiment.load_labels("data/labels/labels.csv")
raw_images, skipped = experiment.list_images(RAW)
generated = experiment.generate_from_clean(RAW, GEN, conditions=("c1", "c3", "c6", "c10"), products=["p01"])
images = raw_images + generated

print(f"전체 {len(images)}장 (직접 찍은 사진 {len(raw_images)}장 + 새우깡 합성 {len(generated)}장)")
for pid, n in sorted(Counter(experiment.parse_name(p)[0] for p in images).items()):
    print(f"  {pid}: {n}장, 정답 라벨 {list(labels.get(pid, {}))}")
if skipped:
    print("[건너뜀]", ", ".join(skipped))

## 4. 사진 품질 진단 미리보기 (OCR 없이 금방 끝남)

`retake`가 `True`면 "다시 찍어 주세요" 안내가 나간다는 뜻입니다. 멀쩡한 사진에 `True`가 나오면 기준이 틀린 것입니다.

In [ ]:
rows = []
for p in images:
    q = preprocess.assess_quality(preprocess.load_image(p))
    m = q["metrics"]
    rows.append({"file": p.name, "brightness": m["brightness"], "sharpness_index": m["sharpness_index"],
                 "text_height": m["text_height"], "issues": ",".join(q["issues"]) or "-", "retake": q["retake"]})
quality = pd.DataFrame(rows)
quality

## 5. OCR 불러오기

In [ ]:
from ocr_eval import OCREngine

ocr = OCREngine(OCR_ENGINE)
print("OCR 준비 완료:", OCR_ENGINE)

## 6. 비교 실험 (1시간에서 1시간 30분)

사진마다 `none`과 `auto` 두 가지로 OCR을 돌립니다. `[1/43]`부터 `[43/43]`까지 한 줄씩 늘어나면 정상입니다.

In [ ]:
df = experiment.run_comparison(images, ocr, labels, OUT, variants=("none", "auto"))
print("\n저장 위치:", OUT)

## 7. 결과 보기

- `cer_mean`: 오류율(낮을수록 좋음). 정답 라벨이 있는 정보만 계산합니다.
- `mean_score`: OCR 확신도(0에서 1). 재촬영 기준을 다시 정할 때 씁니다.

In [ ]:
from IPython.display import display

table = df.pivot_table(index=["product", "condition_name", "file"], columns="variant", values="cer_mean").round(3)
display(table)

auto = df[df.variant == "auto"][["file", "mean_score", "issues", "retake", "rotation", "steps", "sec"]]
display(auto)

## 8. (선택) 방향 판별 모델 속도 실험

뒤집힘 확인을 OCR 대신 PaddleOCR에 들어 있는 **방향 판별 모델**로 하면 얼마나 빠르고 정확한지 봅니다. 오류가 나면 이 셀은 건너뛰어도 됩니다.

In [ ]:
import time

try:
    from paddleocr import DocImgOrientationClassification
    clf = DocImgOrientationClassification(model_name="PP-LCNet_x1_0_doc_ori")
    rows = []
    for p in images:
        pid, cond, tag = experiment.parse_name(p)
        if cond not in ("c0", "c6"):
            continue
        img = preprocess.load_image(p)
        start = time.time()
        res = list(clf.predict(img, batch_size=1))[0]
        sec = time.time() - start
        data = res.json if hasattr(res, "json") else dict(res)
        data = data.get("res", data)
        rows.append({"file": p.name, "predicted": (data.get("label_names") or ["?"])[0],
                     "score": round(float((data.get("scores") or [0])[0]), 3), "sec": round(sec, 2)})
    orientation = pd.DataFrame(rows)
    orientation.to_csv(OUT / "orientation_classifier.csv", index=False, encoding="utf-8-sig")
    display(orientation)
except Exception as e:
    print("방향 판별 모델 실험을 건너뜁니다:", repr(e)[:300])

## 9. 결과 내려받기

`ai2_week3_paddle.zip`이 내려받아집니다. 다운로드가 안 됐으면 이 셀만 다시 실행하세요.

In [ ]:
import shutil
from google.colab import files

quality.to_csv(OUT / "quality_preview.csv", index=False, encoding="utf-8-sig")
shutil.make_archive("/content/ai2_week3_paddle", "zip", OUT)
files.download("/content/ai2_week3_paddle.zip")